In [2]:
# ============================================================
# MiniCheck Evaluation on SAME DeBERTa Test CSV
# Kaggle - One Cell
# ============================================================

!pip -q install "minicheck @ git+https://github.com/Liyan06/MiniCheck.git@main" scikit-learn pandas tqdm

import os
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)
from minicheck.minicheck import MiniCheck

# =========================
# CONFIG
# =========================
TEST_CSV = "/kaggle/input/datasets/belalheshamfathy/tttttttttt/test_predictions _oragnail.csv"

OUTPUT_DIR = "/kaggle/working/minicheck_same_testset_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Start with 1000 for quick test, then set to None for full 6000
MAX_SAMPLES = None
# MAX_SAMPLES = None

# Stronger but heavier
MINICHECK_MODEL = "flan-t5-large"

# If GPU memory is weak, use this instead:
# MINICHECK_MODEL = "deberta-v3-large"

BATCH_SIZE = 8

# =========================
# LOAD TEST CSV
# =========================
df = pd.read_csv(TEST_CSV)

required_cols = ["knowledge", "question", "answer", "label"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df[required_cols].copy()
df["label"] = df["label"].astype(int)

print("===== Loaded Test CSV =====")
print("Total samples:", len(df))
print("Columns:", df.columns.tolist())
print("Label distribution:", df["label"].value_counts().to_dict())

# Optional subset for quick run
if MAX_SAMPLES is not None:
    df_0 = df[df["label"] == 0].sample(
        n=MAX_SAMPLES // 2,
        random_state=42
    )
    df_1 = df[df["label"] == 1].sample(
        n=MAX_SAMPLES // 2,
        random_state=42
    )
    df = pd.concat([df_0, df_1]).sample(frac=1, random_state=42).reset_index(drop=True)

print("\n===== Evaluation Set =====")
print("Total evaluated:", len(df))
print("Label distribution:", df["label"].value_counts().to_dict())

# =========================
# CLAIM FORMAT
# =========================
def make_claim(question, answer):
    return f"Question: {question}\nAnswer: {answer}"

df["claim"] = [
    make_claim(q, a)
    for q, a in zip(df["question"].astype(str), df["answer"].astype(str))
]

docs = df["knowledge"].fillna("").astype(str).tolist()
claims = df["claim"].fillna("").astype(str).tolist()
gold = df["label"].astype(int).tolist()

# =========================
# LOAD MINICHECK
# =========================
print("\nLoading MiniCheck...")
scorer = MiniCheck(model_name=MINICHECK_MODEL, cache_dir="./ckpts")
print("MiniCheck loaded:", MINICHECK_MODEL)

# =========================
# RUN MINICHECK
# =========================
all_preds = []
all_probs = []

for start in tqdm(range(0, len(df), BATCH_SIZE), desc="Running MiniCheck"):
    end = start + BATCH_SIZE

    batch_docs = docs[start:end]
    batch_claims = claims[start:end]

    pred_label, raw_prob, _, _ = scorer.score(
        docs=batch_docs,
        claims=batch_claims
    )

    all_preds.extend([int(x) for x in pred_label])
    all_probs.extend([float(x) for x in raw_prob])

df["minicheck_prediction"] = all_preds
df["minicheck_supported_prob"] = all_probs
df["minicheck_correct"] = df["label"] == df["minicheck_prediction"]

# =========================
# METRICS
# =========================
acc = accuracy_score(gold, all_preds)

macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    gold,
    all_preds,
    average="macro",
    zero_division=0
)

weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
    gold,
    all_preds,
    average="weighted",
    zero_division=0
)

cm = confusion_matrix(gold, all_preds, labels=[0, 1])

report = classification_report(
    gold,
    all_preds,
    labels=[0, 1],
    target_names=["Hallucinated", "Supported"],
    zero_division=0
)

print("\n" + "="*70)
print("MiniCheck Evaluation on Same DeBERTa Test Set")
print("="*70)
print(f"Model             : {MINICHECK_MODEL}")
print(f"Total evaluated   : {len(df)}")
print(f"Accuracy          : {acc:.4f}")
print(f"Macro Precision   : {macro_precision:.4f}")
print(f"Macro Recall      : {macro_recall:.4f}")
print(f"Macro F1          : {macro_f1:.4f}")
print(f"Weighted F1       : {weighted_f1:.4f}")

print("\nConfusion Matrix labels = [Hallucinated(0), Supported(1)]")
print(cm)

print("\nClassification Report:")
print(report)

# =========================
# SAVE OUTPUTS
# =========================
pred_path = os.path.join(OUTPUT_DIR, "minicheck_test_predictions.csv")
summary_path = os.path.join(OUTPUT_DIR, "minicheck_summary.json")

df.to_csv(pred_path, index=False)

summary = {
    "method": "MiniCheck",
    "model": MINICHECK_MODEL,
    "test_file": TEST_CSV,
    "total_evaluated": int(len(df)),
    "accuracy": float(acc),
    "macro_precision": float(macro_precision),
    "macro_recall": float(macro_recall),
    "macro_f1": float(macro_f1),
    "weighted_precision": float(weighted_precision),
    "weighted_recall": float(weighted_recall),
    "weighted_f1": float(weighted_f1),
    "confusion_matrix_labels": ["Hallucinated(0)", "Supported(1)"],
    "confusion_matrix": cm.tolist()
}

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\nSaved files:")
print(pred_path)
print(summary_path)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
===== Loaded Test CSV =====
Total samples: 6000
Columns: ['knowledge', 'question', 'answer', 'label']
Label distribution: {0: 3000, 1: 3000}

===== Evaluation Set =====
Total evaluated: 6000
Label distribution: {0: 3000, 1: 3000}

Loading MiniCheck...


Loading weights:   0%|          | 0/560 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


MiniCheck loaded: flan-t5-large


Running MiniCheck:   0%|          | 0/750 [00:00<?, ?it/s]


Evaluating: 100%|██████████| 8/8 [00:00<00:00,  8.68it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00,  9.52it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00,  9.50it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00,  9.70it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00,  9.77it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00, 10.22it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00,  9.87it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00,  9.73it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00, 10.31it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00,  9.52it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00,  9.61it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00, 10.13it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00, 10.21it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00, 10.04it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00, 10.11it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00,  9.66it/s]

Evaluating: 100%|██████████| 8/8 [00:00<00:00,  9.93it/


MiniCheck Evaluation on Same DeBERTa Test Set
Model             : flan-t5-large
Total evaluated   : 6000
Accuracy          : 0.7378
Macro Precision   : 0.7381
Macro Recall      : 0.7378
Macro F1          : 0.7377
Weighted F1       : 0.7377

Confusion Matrix labels = [Hallucinated(0), Supported(1)]
[[2267  733]
 [ 840 2160]]

Classification Report:
              precision    recall  f1-score   support

Hallucinated       0.73      0.76      0.74      3000
   Supported       0.75      0.72      0.73      3000

    accuracy                           0.74      6000
   macro avg       0.74      0.74      0.74      6000
weighted avg       0.74      0.74      0.74      6000


Saved files:
/kaggle/working/minicheck_same_testset_outputs/minicheck_test_predictions.csv
/kaggle/working/minicheck_same_testset_outputs/minicheck_summary.json
